In [1]:
# If FEVER dataset not present, download

from urllib.request import urlretrieve
import os
from zipfile import ZipFile

# Download paper development set
if not os.path.exists("paper_dev.jsonl"):
    url = "https://fever.ai/download/fever/paper_dev.jsonl"
    filename = "paper_dev.jsonl"
    urlretrieve(url, filename)
    print("Downloaded Paper Development Set")
else:
    print("Paper Development Set already downloaded")

# Download wiki dump
if not os.path.exists("wiki-pages.zip"):
    url = "https://fever.ai/download/fever/wiki-pages.zip"
    filename = "wiki-pages.zip"
    urlretrieve(url, filename)
    print("Downloaded Wiki Pages Dump")
else:
    print("Wiki Pages Dump already downloaded")

# Unzip wiki dump
if not os.path.exists("wiki-pages"):
    zip_file_path = "wiki-pages.zip"
    extraction_path = "wiki-pages"
    with ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall(extraction_path)
    print("Wiki Pages Dump zipfile extracted")
else:
    print("Wiki Pages Dump already extracted")

Paper Development Set already downloaded
Wiki Pages Dump already downloaded
Wiki Pages Dump already extracted


In [2]:
# Load development dataset using Hugging Face

from datasets import load_dataset

data_files = {
    "development": "paper_dev.jsonl"
}

dataset = load_dataset(path="json", data_files=data_files)
print(dataset)

/home/ubuntu/projects/urban-dollop/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    development: Dataset({
        features: ['id', 'verifiable', 'label', 'claim', 'evidence'],
        num_rows: 9999
    })
})


In [3]:
# Load dump using Hugging Face

data_files = {
    "dump": "wiki-pages/wiki-pages/*.jsonl"
}

dump = load_dataset(path="json", data_files=data_files, split="dump")
print(dump)

Dataset({
    features: ['id', 'text', 'lines'],
    num_rows: 5416537
})


In [4]:
# Create evidence look up index for dump

import json

dump_lookup_index = {}
filename = "dump_lookup_index.json"

# If index doesn't exist, create then save
if not os.path.exists(filename):
    print("Dump lookup index does not exist.")
    print("Creating index...")
    for value, key in enumerate(dump['id']):
        dump_lookup_index[key] = value
        # Track progress
        if value % 10000 == 0:
            print(key, value)
    print("Saving index...")
    with open(filename, "w") as file:
        json.dump(dump_lookup_index, file)
else:
    print("Dump lookup index exists.")
    print("Loading index...")
    with open("dump_lookup_index.json", "r") as file:
        dump_lookup_index = json.load(file)



Dump lookup index exists.
Loading index...


In [5]:
# Construct pairs for cross-encoder, concatenate evidence

def construct_pair(row):
    evidences = []
    for evidence in row['evidence']:
        for annotation in evidence:
            wiki_url = annotation[2]
            line_idx = annotation[3]
            dump_id = dump_lookup_index.get(wiki_url, None)
            if dump_id is None:
                continue
            annotation_text = dump[dump_id]['lines'].splitlines()[line_idx]
            evidences.append(annotation_text)
    if not evidences:
        evidences_concatenated = ""
    else:
        evidences_concatenated = "\n".join(evidences)
    pair = (row['claim'], evidences_concatenated)
    return pair

# Note: naively falls back to empty string if wiki_url is not present in lookup index. Should account for this in later refinement
        

In [6]:
# Load cross-encoder

# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch

# model_name = "cross-encoder/nli-deberta-v3-base"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name)
# model.eval()

# def use_cross_encoder(claims, evidences):
#     features = tokenizer(claims, evidences, padding=True, truncation=True, return_tensors="pt")
#     with torch.no_grad():
#         scores = model(**features).logits
#         label_mapping = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]
#         labels = [label_mapping[score_max] for score_max in scores.argmax(dim=1)]
#     return labels

from sentence_transformers import CrossEncoder

model = CrossEncoder('cross-encoder/nli-deberta-v3-base')
def use_cross_encoder(pairs):
    scores = model.predict(pairs)
    # Convert scores to labels
    label_mapping = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]
    labels = [label_mapping[score_max] for score_max in scores.argmax(axis=1)]
    return labels



# Topics to study:
# What is the tokenizer method? What is padding, truncation, and what is meant by return_tensors? Does this method return tensors of PyTorch?
# What is model.eval()?
# What is torch.no_grad()?
# What does model(...) produce? Are they tensors? If there is an option for logits, what else is there? A probability option?
# What is this .argmax(...) method for scores? Is this a method for tensors? What is meant by specifying a dimension of 1?



/home/ubuntu/projects/urban-dollop/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 202/202 [00:00<00:00, 203.39it/s]


In [ ]:
# Compute batch of pairs, then process through cross-encoder. Then store result in predictions

predictions = []
# for row in dataset['development']:
#     pair = [construct_pair(row)]
#     prediction = use_cross_encoder(pair)
#     predictions.extend(prediction)
#     print(len(predictions), "Added:", prediction)

batch = []
batch_limit = 2
for row in dataset['development']:
    pair = construct_pair(row)
    batch.append(pair)
    if len(batch) < batch_limit:
        continue
    batch_predictions = use_cross_encoder(batch)
    predictions.extend(batch_predictions)
    print(len(predictions), "Added:", batch_predictions)
    batch = []
if batch:
    batch_predictions = use_cross_encoder(batch)
    predictions.extend(batch_predictions)
    print(len(predictions), "Added:", batch_predictions)


2 Added: ['NOT ENOUGH INFO', 'NOT ENOUGH INFO']
4 Added: ['REFUTES', 'SUPPORTS']
6 Added: ['NOT ENOUGH INFO', 'REFUTES']
8 Added: ['REFUTES', 'REFUTES']
10 Added: ['SUPPORTS', 'SUPPORTS']
12 Added: ['NOT ENOUGH INFO', 'NOT ENOUGH INFO']
14 Added: ['REFUTES', 'REFUTES']
16 Added: ['NOT ENOUGH INFO', 'SUPPORTS']
18 Added: ['REFUTES', 'SUPPORTS']
20 Added: ['NOT ENOUGH INFO', 'SUPPORTS']
22 Added: ['REFUTES', 'SUPPORTS']
24 Added: ['NOT ENOUGH INFO', 'NOT ENOUGH INFO']
26 Added: ['NOT ENOUGH INFO', 'REFUTES']
28 Added: ['NOT ENOUGH INFO', 'SUPPORTS']
30 Added: ['SUPPORTS', 'SUPPORTS']
32 Added: ['REFUTES', 'REFUTES']
34 Added: ['NOT ENOUGH INFO', 'REFUTES']
36 Added: ['REFUTES', 'REFUTES']
38 Added: ['SUPPORTS', 'NOT ENOUGH INFO']
40 Added: ['NOT ENOUGH INFO', 'NOT ENOUGH INFO']
42 Added: ['REFUTES', 'REFUTES']
44 Added: ['SUPPORTS', 'REFUTES']
46 Added: ['NOT ENOUGH INFO', 'REFUTES']
48 Added: ['SUPPORTS', 'REFUTES']
50 Added: ['NOT ENOUGH INFO', 'SUPPORTS']
52 Added: ['NOT ENOUGH INFO',

In [10]:
# Save predictions
filename = "predictions-v1.json"
if not os.path.exists(filename):
    print("Saving predictions...")
    with open(filename, "w") as file:
        json.dump(predictions, file)

Saving predictions...


In [11]:
# Load predictions
if os.path.exists(filename):
    print("Loading predictions...")
    with open(filename, "r") as file:
        predictions = json.load(file)

Loading predictions...


In [17]:
# Compute count of each class prediction

from pprint import pprint

prediction_class_count = {"name": "predictions_class_count", "total_count": len(predictions)}
expected_class_count = {"name": "expected_class_count", "total_count": len(dataset['development'])}
label_mapping = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]
for label in label_mapping:
    prediction_class_count[label] = predictions.count(label)
    expected_class_count[label] = dataset['development']['label'].count(label)

pprint(prediction_class_count)
pprint(expected_class_count)

{'NOT ENOUGH INFO': 4283,
 'REFUTES': 3035,
 'SUPPORTS': 2680,
 'name': 'predictions_class_count',
 'total_count': 9998}
{'NOT ENOUGH INFO': 3333,
 'REFUTES': 3333,
 'SUPPORTS': 3333,
 'name': 'expected_class_count',
 'total_count': 9999}


In [ ]:
# Compare results, compute recall, precision, F1, confusion matrix for each class